In [27]:
# imports
%matplotlib qt
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import find_peaks, peak_prominences, peak_widths

mne.viz.set_browser_backend("qt")


def need(*names):  # проверяет, что нужные переменные уже созданы
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError(f"Нужны переменные: {missing}")


def as_float(value, default=np.nan):  # переводит значение в число или default
    try:
        value = float(value)
    except Exception:
        return default
    return value if np.isfinite(value) else default


def idx_at(axis, value):  # находит индекс ближайшего времени
    return int(np.argmin(np.abs(axis - value)))


In [28]:
# загрузка MATLAB-записи из папки проекта data/raw

project_root = Path.cwd()  # папка запуска ноутбука
if project_root.name == "notebooks":
    project_root = project_root.parent

outputs_dir = project_root / "outputs"  # результаты
annotations_dir = outputs_dir / "annotations"  # аннотации
figures_dir = outputs_dir / "figures"  # графики

for folder in [annotations_dir, figures_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# загрузка из папки проекта data/raw
data_raw = project_root / "data" / "raw"
mat_path = data_raw / "pig_4_9_14_day_after_SCI.mat"

if not mat_path.exists():
    raise FileNotFoundError(f"Файл не найден: {mat_path}")

# MATLAB хранит каналы каждого из четырёх блоков последовательно в data.
mat = loadmat(
    mat_path,
    variable_names=["data", "datastart", "dataend", "titles", "samplerate"],
    simplify_cells=True,
)
data = mat["data"].ravel()
starts = mat["datastart"].astype(int) - 1  # индексы MATLAB начинаются с 1
ends = mat["dataend"].astype(int)
ch_names = [name.strip() for name in mat["titles"]]
sfreq = float(mat["samplerate"][0, 0])

blocks = []
for block_idx in range(starts.shape[1]):
    block_data = np.vstack([
        data[starts[ch_idx, block_idx]:ends[ch_idx, block_idx]]
        for ch_idx in range(len(ch_names))
    ])
    info = mne.create_info(ch_names, sfreq, ch_types="emg")
    blocks.append(mne.io.RawArray(block_data, info, verbose=False))

# Сохраняем границы исходных блоков как boundary-аннотации MNE.
raw = mne.concatenate_raws(blocks, preload=True, verbose=False)
raw_original = raw.copy()

# параметры влияют только на отображение, данные остаются в вольтах
emg_browser_kwargs = dict(
    duration=1.0,
    n_channels=min(8, len(raw.ch_names)),
    scalings={"emg": 8.0779357},  # V; 8077935.7 µV from the browser settings
)

sfreq = raw.info["sfreq"]
duration_s = raw.n_times / sfreq

print("File:", mat_path.name)
print("Channels:", len(raw.ch_names))
print("sfreq:", sfreq)
print("Duration, s:", round(duration_s, 2))
print("First channels:", raw.ch_names[:10])


File: pig_4_9_14_day_after_SCI.mat
Channels: 7
sfreq: 4000.0
Duration, s: 538.4
First channels: ['VL Left', 'TA Left', 'GM left', 'TA Right', 'GM Right', 'Art', 'Art 2']


In [29]:
# посмотреть сырую запись в MNE 
raw.plot(**emg_browser_kwargs, block=False)


Using pyopengl with version 3.1.10


In [30]:
# авторазметка стимулов

stim_ch = "Art"
stim_label = "Stimulus_Auto"
analysis_ch = "GM Right"
expected_isi_ms = 500.0
isi_tolerance_ms = 10.0
stim_noise_k = 3.0  # permissive threshold; cadence filters false candidates
max_cadence_gap_ms = 20000.0  # longest supported gap inside a train

missing_channels = [ch for ch in [stim_ch, analysis_ch] if ch not in raw.ch_names]
if missing_channels:
    raise ValueError(f"Каналы не найдены: {missing_channels}")

sfreq = raw.info["sfreq"]
expected_isi_samples = int(round(expected_isi_ms / 1000 * sfreq))
isi_tolerance_samples = int(round(isi_tolerance_ms / 1000 * sfreq))
stim_min_distance_samples = expected_isi_samples - isi_tolerance_samples
if stim_min_distance_samples <= 0:
    raise ValueError("Допуск ISI должен быть меньше ожидаемого ISI")

art = raw.copy().pick([stim_ch]).get_data()[0]
art_abs = np.abs(art - np.median(art))
art_median = np.median(art_abs)
art_noise = 1.4826 * np.median(np.abs(art_abs - art_median))
peak_threshold = art_median + stim_noise_k * art_noise

# Scan the concatenated recording globally. Cadence is checked locally, so separate
# stimulation trains may start at different phases without using MATLAB block IDs.
threshold_samples, _ = find_peaks(art_abs, height=peak_threshold)
candidate_samples, candidate_properties = find_peaks(
    art_abs, height=peak_threshold, distance=stim_min_distance_samples
)
max_cadence_steps = int(round(max_cadence_gap_ms / expected_isi_ms))

cadence_supported = []
for sample in candidate_samples:
    deltas = np.abs(candidate_samples - sample)
    steps = np.rint(deltas / expected_isi_samples).astype(int)
    cadence_error = np.abs(deltas - steps * expected_isi_samples)
    has_neighbor = np.any(
        (steps >= 1) & (steps <= max_cadence_steps)
        & (cadence_error <= isi_tolerance_samples)
    )
    cadence_supported.append(has_neighbor)
cadence_samples = candidate_samples[np.asarray(cadence_supported, dtype=bool)]

# Split global candidates into local cadence trains; no MATLAB block information is used.
cadence_trains = []
if len(cadence_samples):
    train_start = 0
    for idx, gap in enumerate(np.diff(cadence_samples)):
        gap_steps = int(round(gap / expected_isi_samples))
        gap_error = abs(gap - gap_steps * expected_isi_samples)
        if gap_steps < 1 or gap_steps > max_cadence_steps or gap_error > 2 * isi_tolerance_samples:
            cadence_trains.append(cadence_samples[train_start:idx + 1])
            train_start = idx + 1
    cadence_trains.append(cadence_samples[train_start:])
cadence_trains = [train for train in cadence_trains if len(train) >= 2]

cadence_added_samples = []
final_parts = []
for train in cadence_trains:
    final_parts.append(train)
    for left, right in zip(train[:-1], train[1:]):
        n_steps = int(round((right - left) / expected_isi_samples))
        for step in range(1, n_steps):
            predicted = left + step * expected_isi_samples
            lo = max(0, predicted - isi_tolerance_samples)
            hi = min(len(art_abs), predicted + isi_tolerance_samples + 1)
            recovered = lo + int(np.argmax(art_abs[lo:hi]))
            cadence_added_samples.append(recovered)

if cadence_added_samples:
    final_parts.append(np.asarray(cadence_added_samples, dtype=int))
stim_samples = np.unique(np.concatenate(final_parts)).astype(int) if final_parts else np.array([], dtype=int)
if len(stim_samples) == 0:
    raise RuntimeError(f"Не найдено ни одного стимула на {stim_ch}")

funnel_df = pd.DataFrame([
    {"stage": "Amplitude threshold", "output_count": len(threshold_samples), "removed": np.nan, "added": 0},
    {"stage": "Minimum ISI distance", "output_count": len(candidate_samples), "removed": len(threshold_samples) - len(candidate_samples), "added": 0},
    {"stage": "Local cadence support", "output_count": len(cadence_samples), "removed": len(candidate_samples) - len(cadence_samples), "added": 0},
    {"stage": "Cadence gap completion", "output_count": len(stim_samples), "removed": 0, "added": len(stim_samples) - len(cadence_samples)},
    {"stage": "Final annotations", "output_count": len(stim_samples), "removed": 0, "added": 0},
])

# Audit saved manual QC labels against every detector stage.
manual_qc_path = annotations_dir / "stimulus_manual_qc-annot.fif"
manual_audit_rows = []
if manual_qc_path.exists():
    manual_qc = mne.read_annotations(manual_qc_path)
    manual_mask = np.char.find(np.asarray(manual_qc.description, dtype=str), "missing") >= 0
    manual_samples = np.rint(np.asarray(manual_qc.onset)[manual_mask] * sfreq).astype(int)

    def stage_has_sample(stage_samples, sample):
        return bool(len(stage_samples) and np.min(np.abs(stage_samples - sample)) <= isi_tolerance_samples)

    for sample in manual_samples:
        lo = max(0, sample - isi_tolerance_samples)
        hi = min(len(art_abs), sample + isi_tolerance_samples + 1)
        strength_x_noise = (np.max(art_abs[lo:hi]) - art_median) / art_noise
        passes = {
            "amplitude": stage_has_sample(threshold_samples, sample),
            "min_distance": stage_has_sample(candidate_samples, sample),
            "cadence_support": stage_has_sample(cadence_samples, sample),
            "final": stage_has_sample(stim_samples, sample),
        }
        first_failed = next((name for name, passed in passes.items() if not passed), "none")
        manual_audit_rows.append({
            "time_s": sample / sfreq, "strength_x_noise": strength_x_noise,
            **passes, "first_failed_stage": first_failed,
        })
manual_audit_df = pd.DataFrame(manual_audit_rows)
old_ann = raw.annotations
keep = np.asarray(old_ann.description) != stim_label
clean_ann = mne.Annotations(old_ann.onset[keep], old_ann.duration[keep], np.asarray(old_ann.description)[keep], orig_time=old_ann.orig_time)
stim_ann = mne.Annotations(stim_samples / sfreq, np.zeros(len(stim_samples)), [stim_label] * len(stim_samples), orig_time=old_ann.orig_time)
raw.set_annotations(clean_ann + stim_ann)

print(f"Threshold {stim_ch} abs ({stim_noise_k:.1f} x noise): {peak_threshold:.6g}")
print("Expected/tolerance/min distance, samples:", expected_isi_samples, isi_tolerance_samples, stim_min_distance_samples)
print("Cadence trains (start, stop, count):")
display(pd.DataFrame([
    {"start_s": train[0] / sfreq, "stop_s": train[-1] / sfreq, "anchors": len(train)}
    for train in cadence_trains
]))
display(funnel_df)
if len(manual_audit_df):
    print("Saved manual-label audit:")
    display(manual_audit_df)
print("Stimulus_Auto count:", int(np.sum(raw.annotations.description == stim_label)))


Threshold Art abs (3.0 x noise): 2.35753
Expected/tolerance/min distance, samples: 2000 40 1960
Cadence trains (start, stop, count):


,start_s,stop_s,anchors
0,81.18575,194.1835,195
1,252.99300,335.4885,130
2,381.40450,465.4030,169


,stage,output_count,removed,added
0,Amplitude threshold,2221,NaN,0
1,Minimum ISI distance,500,1721.0,0
2,Local cadence support,494,6.0,0
3,Cadence gap completion,562,0.0,68
4,Final annotations,562,0.0,0


Saved manual-label audit:


,time_s,strength_x_noise,amplitude,min_distance,cadence_support,final,first_failed_stage
0,117.68350,2.729833,False,False,False,True,amplitude
1,118.18350,1.677606,False,False,False,True,amplitude
2,118.68350,1.992125,False,False,False,True,amplitude
3,119.18375,2.051162,False,False,False,True,amplitude
4,119.68350,1.610732,False,False,False,True,amplitude
5,120.18375,2.066314,False,False,False,True,amplitude
6,120.68400,1.984811,False,False,False,True,amplitude
7,121.18375,2.131098,False,False,False,True,amplitude
8,121.68375,2.091914,False,False,False,True,amplitude
9,317.99100,2.522940,False,False,False,True,amplitude


Stimulus_Auto count: 562


In [31]:
# QC: raw browser со всеми каналами и Stimulus_Auto-аннотациями

stim_times_s = (stim_samples - raw.first_samp) / raw.info["sfreq"]
qc_start_s = max(0.0, float(stim_times_s[0] - 0.5))
print(f"Opening raw browser at {qc_start_s:.3f} s with {len(raw.ch_names)} channels.")
raw.plot(
    start=qc_start_s,
    duration=5.0,
    n_channels=len(raw.ch_names),
    scalings=emg_browser_kwargs["scalings"],
    title="Stimulus QC — all channels with Stimulus_Auto annotations",
    block=False,
)


Opening raw browser at 80.686 s with 7 channels.
Using pyopengl with version 3.1.10


In [32]:
# Распределение ISI

all_isi_ms = np.diff(stim_samples) / sfreq * 1000
within_train_mask = all_isi_ms <= max_cadence_gap_ms
isi_ms = all_isi_ms[within_train_mask]
if not len(isi_ms):
    raise RuntimeError("Недостаточно стимулов для расчёта ISI")
isi_multiples = np.maximum(1, np.rint(isi_ms / expected_isi_ms)).astype(int)
isi_residual_ms = isi_ms - isi_multiples * expected_isi_ms

print("Accepted stimuli:", len(stim_samples))
print("Within-train ISI median/min/max, ms:", np.median(isi_ms), np.min(isi_ms), np.max(isi_ms))
print("Between-train gaps excluded:", int((~within_train_mask).sum()))
print("Cadence residual min/max, ms:", np.min(isi_residual_ms), np.max(isi_residual_ms))
print("Intervals outside tolerance:", int(np.sum(np.abs(isi_residual_ms) > isi_tolerance_ms)))

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(isi_ms, bins=50)
axes[0].set(xlabel="Within-train ISI, ms", ylabel="count", title="Accepted stimulus intervals")
axes[1].hist(isi_residual_ms, bins=np.arange(-isi_tolerance_ms - 0.5, isi_tolerance_ms + 1.0, 0.5))
axes[1].axvline(-isi_tolerance_ms, color="red", linestyle="--")
axes[1].axvline(isi_tolerance_ms, color="red", linestyle="--")
axes[1].set(xlabel="Residual from 500 ms cadence, ms", ylabel="count", title="Cadence residuals")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Accepted stimuli: 562
Within-train ISI median/min/max, ms: 500.0 490.25 509.75000000000006
Between-train gaps excluded: 2
Cadence residual min/max, ms: -9.75 9.750000000000057
Intervals outside tolerance: 0


In [33]:
# нарезка эпох

stim_label = "Stimulus_Auto"
tmin = -0.030  # enough pre-stimulus data for a stable noise baseline
tmax = 0.490  # stop before the earliest allowed next stimulus
min_next_stim_interval_ms = expected_isi_ms - isi_tolerance_ms

events_auto, event_id_auto = mne.events_from_annotations(raw, regexp=stim_label)
stim_id_auto = event_id_auto[stim_label]
stim_events = events_auto[events_auto[:, 2] == stim_id_auto]
stim_samples_auto = stim_events[:, 0]
isi_ms = np.diff(stim_samples_auto) / raw.info["sfreq"] * 1000
valid_isi_mask = isi_ms >= min_next_stim_interval_ms
valid_events = stim_events[:-1][valid_isi_mask]

epochs_metadata = pd.DataFrame({
    "epoch_index": np.arange(len(valid_events)),
    "stim_sample": valid_events[:, 0],
    "next_stim_sample": stim_samples_auto[1:][valid_isi_mask],
    "stim_time_s": valid_events[:, 0] / raw.info["sfreq"],
    "isi_ms": isi_ms[valid_isi_mask],
})

epochs_auto = mne.Epochs(
    raw_original,
    events=valid_events,
    event_id={stim_label: stim_id_auto},
    tmin=tmin,
    tmax=tmax,
    baseline=None,
    preload=True,
    reject_by_annotation=False,
    metadata=epochs_metadata,
)

print(epochs_auto)
print("Original stimuli:", len(stim_samples_auto))
print("Valid epochs:", len(epochs_auto))
print("Dropped by minimum-ISI filter:", int((~valid_isi_mask).sum()))
print("Epoch data shape:", epochs_auto.get_data().shape)
print("Epoch time range, ms:", epochs_auto.times[[0, -1]] * 1000)
display(epochs_auto.metadata.head())


Used Annotations descriptions: [np.str_('Stimulus_Auto')]
Adding metadata with 5 columns
561 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 561 events and 2081 original time points ...
0 bad epochs dropped
<Epochs | 561 events (all good), -0.03 – 0.49 s (baseline off), ~62.4 MB, data loaded, with metadata,
 'Stimulus_Auto': 561>
Original stimuli: 562
Valid epochs: 561
Dropped by minimum-ISI filter: 0
Epoch data shape: (561, 7, 2081)
Epoch time range, ms: [-30. 490.]


,epoch_index,stim_sample,next_stim_sample,stim_time_s,isi_ms
0,0,324743,326743,81.18575,500.0
1,1,326743,328743,81.68575,500.0
2,2,328743,330743,82.18575,500.0
3,3,330743,332743,82.68575,500.0
4,4,332743,334743,83.18575,500.0


In [34]:
# подготовка данных канала GM Right

ch_name = analysis_ch
if ch_name not in epochs_auto.ch_names:
    raise ValueError(f"Канал {ch_name!r} не найден в epochs_auto")

amplitude_unit = "native"  # MAT physical calibration is not documented
epochs_data_uv = epochs_auto.get_data(picks=[ch_name])[:, 0, :]
times_ms = epochs_auto.times * 1000
epochs_meta = epochs_auto.metadata.copy().reset_index(drop=True)

print("Analysis channel:", ch_name)
print(f"epochs_data shape ({amplitude_unit} units):", epochs_data_uv.shape)
print("times_ms range:", times_ms[0], "to", times_ms[-1])
print("metadata shape:", epochs_meta.shape)
display(epochs_meta.head())


Analysis channel: GM Right
epochs_data shape (native units): (561, 2081)
times_ms range: -30.0 to 490.0
metadata shape: (561, 5)


,epoch_index,stim_sample,next_stim_sample,stim_time_s,isi_ms
0,0,324743,326743,81.18575,500.0
1,1,326743,328743,81.68575,500.0
2,2,328743,330743,82.18575,500.0
3,3,330743,332743,82.68575,500.0
4,4,332743,334743,83.18575,500.0


In [35]:
# baseline шума для каждой эпохи

baseline_window_ms = (-25.0, -5.0)
baseline_mask = (times_ms >= baseline_window_ms[0]) & (times_ms <= baseline_window_ms[1])

if not baseline_mask.any():
    raise ValueError("Baseline window не попал во временную ось epochs_auto")

baseline_data = epochs_data_uv[:, baseline_mask]
baseline_median_uv = np.median(baseline_data, axis=1)
baseline_noise_uv = 1.4826 * np.median(np.abs(baseline_data - baseline_median_uv[:, None]), axis=1)
baseline_noise_outlier_k = 5.0
baseline_noise_reference_uv = float(np.median(baseline_noise_uv))
baseline_noise_outlier = baseline_noise_uv > baseline_noise_outlier_k * baseline_noise_reference_uv
baseline_df = epochs_meta.assign(
    baseline_median_uv=baseline_median_uv,
    baseline_noise_uv=baseline_noise_uv,
    baseline_noise_outlier=baseline_noise_outlier,
)

print("Baseline samples:", int(baseline_mask.sum()))
print("Baseline window, ms:", times_ms[baseline_mask][[0, -1]])
display(baseline_df[["baseline_median_uv", "baseline_noise_uv"]].describe())
print(f"Noise percentiles (50/90/95/99/max), {amplitude_unit}:", np.percentile(baseline_noise_uv, [50, 90, 95, 99, 100]))
print(f"Noise outliers > {baseline_noise_outlier_k:.1f} x median:", int(baseline_noise_outlier.sum()))

plt.figure(figsize=(8, 3))
plt.hist(baseline_noise_uv, bins=50)
plt.axvline(baseline_noise_reference_uv, color="black", linestyle="--", label="median")
plt.axvline(baseline_noise_outlier_k * baseline_noise_reference_uv, color="red", linestyle="--", label="outlier flag")
plt.xlabel(f"Baseline noise, {amplitude_unit}")
plt.ylabel("count")
plt.title(f"Baseline noise distribution: {ch_name}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


Baseline samples: 81
Baseline window, ms: [-25.  -5.]


,baseline_median_uv,baseline_noise_uv
count,561.000000,561.000000
mean,-0.023460,0.135346
std,0.400493,0.104660
min,-1.300938,0.015753
25%,-0.185312,0.088029
50%,-0.064687,0.120925
75%,0.049375,0.142700
max,3.610312,1.488160


Noise percentiles (50/90/95/99/max), native: [0.12092457 0.18625161 0.29559338 0.53531125 1.48815974]
Noise outliers > 5.0 x median: 4


In [36]:
# 1. Load precalculated ER auto-detector parameters

er_auto_label = "ER_auto"
er_threshold_path = outputs_dir / "er_morphology_thresholds.json"
er_auto_path = annotations_dir / "stimulus_er_mat_auto-annot.fif"

if not er_threshold_path.exists():
    raise FileNotFoundError(f"Precalculated ER parameters not found: {er_threshold_path}")
with er_threshold_path.open(encoding="utf-8") as handle:
    er_parameter_payload = json.load(handle)

required_metrics = [
    "er_start_ms", "er_peak_ms", "er_width_ms",
    "er_full_duration_ms", "er_rebound_depth_ratio",
]
missing_metrics = [name for name in required_metrics if name not in er_parameter_payload.get("metrics", {})]
if missing_metrics:
    raise ValueError(f"Missing ER detector parameters: {missing_metrics}")

er_morphology_limits = {
    name: (float(er_parameter_payload["metrics"][name]["low"]),
           float(er_parameter_payload["metrics"][name]["high"]))
    for name in required_metrics
}
er_metric_stats = {
    name: (float(er_parameter_payload["metrics"][name]["median"]),
           max(float(er_parameter_payload["metrics"][name]["mad_scale"]), np.finfo(float).eps))
    for name in required_metrics
}
ceiling = er_parameter_payload.get("amplitude_ceiling_native")
er_amplitude_ceiling = np.inf if ceiling is None else float(ceiling)
er_threshold_source = str(er_parameter_payload.get("source", "precalculated"))

# Fixed significance settings used with the saved morphology limits.
er_core_frac = 0.50
er_max_prepeak_ms = 6.0
er_max_postpeak_ms = 10.0
er_rebound_window_ms = 20.0
er_min_peak_distance_ms = 0.8
er_depth_snr_min = 3.0
er_p2p_snr_min = 4.0
er_prominence_snr_min = 2.5

if "epoch_index" not in baseline_df.columns:
    baseline_df = baseline_df.assign(epoch_index=np.arange(len(baseline_df)))
baseline_lookup = baseline_df.set_index("epoch_index")
stim_times_s = (epochs_auto.events[:, 0] - raw.first_samp) / raw.info["sfreq"]
dt_ms = float(np.median(np.diff(times_ms)))
global_noise_uv = as_float(np.nanmedian(baseline_df["baseline_noise_uv"]), 1.0)
if global_noise_uv <= 0:
    global_noise_uv = 1.0
peak_window = er_morphology_limits["er_peak_ms"]
er_indices = np.where((times_ms >= peak_window[0]) & (times_ms <= peak_window[1]))[0]
if not len(er_indices):
    raise ValueError(f"Saved ER peak window {peak_window} is outside epochs_auto")
min_peak_distance = max(1, int(round(er_min_peak_distance_ms / dt_ms)))


def er_bounds(y, peak_idx, frac=er_core_frac):
    level = -frac * abs(y[peak_idx])
    left_limit = max(0, peak_idx - int(round(er_max_prepeak_ms / dt_ms)))
    right_limit = min(len(y) - 1, peak_idx + int(round(er_max_postpeak_ms / dt_ms)))
    left_hits = np.where(y[left_limit:peak_idx + 1] >= level)[0]
    right_hits = np.where(y[peak_idx:right_limit + 1] >= level)[0]
    if not len(left_hits) or not len(right_hits):
        return None
    left = left_limit + int(left_hits[-1])
    right = peak_idx + int(right_hits[0])
    return None if left >= peak_idx or right <= peak_idx else (left, right)


def morphology_score(features):
    return float(sum(
        abs(features[name] - er_metric_stats[name][0]) / er_metric_stats[name][1]
        for name in required_metrics
    ))


def detect_er(epoch_idx):
    signal = epochs_data_uv[epoch_idx]
    baseline = float(baseline_lookup.loc[epoch_idx, "baseline_median_uv"])
    epoch_noise = float(baseline_lookup.loc[epoch_idx, "baseline_noise_uv"])
    effective_noise = max(epoch_noise if np.isfinite(epoch_noise) and epoch_noise > 0 else global_noise_uv,
                          global_noise_uv)
    y = signal - baseline
    peaks, properties = find_peaks(-y[er_indices], prominence=0, distance=min_peak_distance)
    candidates = []
    for order, local_peak in enumerate(peaks):
        peak_idx = int(er_indices[int(local_peak)])
        depth = float(-y[peak_idx])
        core = er_bounds(y, peak_idx)
        if depth <= 0 or core is None:
            continue
        left, right = core
        rebound_limit = min(times_ms[-1], times_ms[peak_idx] + er_rebound_window_ms)
        rebound_span = np.where((times_ms > times_ms[peak_idx]) & (times_ms <= rebound_limit))[0]
        if not len(rebound_span):
            continue
        rebound_idx = int(rebound_span[np.argmax(y[rebound_span])])
        rebound_change = float(y[rebound_idx] - y[peak_idx])
        features = {
            "er_start_ms": float(times_ms[left]),
            "er_peak_ms": float(times_ms[peak_idx]),
            "er_width_ms": float(times_ms[right] - times_ms[left]),
            "er_full_duration_ms": float(times_ms[rebound_idx] - times_ms[left]),
            "er_rebound_depth_ratio": rebound_change / depth,
        }
        if any(not er_morphology_limits[name][0] <= features[name] <= er_morphology_limits[name][1]
               for name in required_metrics):
            continue
        p2p = float(np.ptp(signal[left:rebound_idx + 1]))
        if p2p > er_amplitude_ceiling:
            continue
        prominence = float(properties["prominences"][order])
        candidates.append({
            **features, "peak_idx": peak_idx, "left_idx": left, "right_idx": right,
            "rebound_idx": rebound_idx, "er_depth_uv": depth, "er_p2p_uv": p2p,
            "er_core_p2p_uv": float(np.ptp(signal[left:right + 1])),
            "er_prominence_uv": prominence, "er_rebound_change_native": rebound_change,
            "er_morphology_score": morphology_score(features),
        })
    if not candidates:
        return {"epoch_index": epoch_idx, "candidate_found": False, "er_found": False,
                "er_detection_source": "no_morphology_candidate"}
    best = min(candidates, key=lambda item: item["er_morphology_score"])
    depth_snr = best["er_depth_uv"] / effective_noise
    p2p_snr = best["er_p2p_uv"] / effective_noise
    prominence_snr = best["er_prominence_uv"] / effective_noise
    significant = (depth_snr >= er_depth_snr_min and p2p_snr >= er_p2p_snr_min
                   and prominence_snr >= er_prominence_snr_min)
    return {
        "epoch_index": epoch_idx, "candidate_found": True, "er_found": significant,
        "er_significant": significant, "stim_time_s": float(stim_times_s[epoch_idx]),
        "er_start_ms": best["er_start_ms"], "er_peak_ms": best["er_peak_ms"],
        "er_end_ms": float(times_ms[best["rebound_idx"]]),
        "er_core_end_ms": float(times_ms[best["right_idx"]]),
        "er_rebound_ms": float(times_ms[best["rebound_idx"]]),
        "er_full_duration_ms": best["er_full_duration_ms"], "er_width_ms": best["er_width_ms"],
        "er_min_uv": float(signal[best["peak_idx"]]), "er_max_uv": float(signal[best["rebound_idx"]]),
        "er_depth_uv": best["er_depth_uv"], "er_p2p_uv": best["er_p2p_uv"],
        "er_core_p2p_uv": best["er_core_p2p_uv"], "er_prominence_uv": best["er_prominence_uv"],
        "er_rebound_change_native": best["er_rebound_change_native"],
        "er_rebound_depth_ratio": best["er_rebound_depth_ratio"],
        "er_morphology_score": best["er_morphology_score"], "effective_noise_uv": effective_noise,
        "er_depth_noise_ratio": depth_snr, "er_p2p_noise_ratio": p2p_snr,
        "er_prominence_noise_ratio": prominence_snr, "amplitude_unit": amplitude_unit,
        "er_threshold_source": er_threshold_source,
        "er_significance_reason": "significant" if significant else "below_snr",
        "er_detection_source": "significant_er" if significant else "candidate_not_significant",
    }


print("Loaded ER parameters:", er_threshold_path)
print("Threshold source:", er_threshold_source)
display(pd.DataFrame(er_morphology_limits, index=["low", "high"]).T)


Loaded ER parameters: /Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/outputs/er_morphology_thresholds.json
Threshold source: matched_auto_manual_robust_mad


,low,high
er_start_ms,10.0,22.000000
er_peak_ms,12.0,24.171700
er_width_ms,3.5,10.000000
er_full_duration_ms,8.0,31.481450
er_rebound_depth_ratio,0.5,2.040653


In [37]:
# 2. Run production ER auto-detection and save ER_auto annotations

er_detect_df = pd.DataFrame([detect_er(epoch_idx) for epoch_idx in range(len(epochs_data_uv))])
er_df = baseline_df.merge(er_detect_df, on="epoch_index", how="left")
for column in ["candidate_found", "er_found", "er_significant"]:
    if column not in er_df:
        er_df[column] = False
    er_df[column] = er_df[column].fillna(False).astype(bool)


def annotations_from_er_rows(rows, label):
    records = []
    for _, row in rows.dropna(subset=["er_start_ms", "er_end_ms"]).iterrows():
        if row["er_end_ms"] <= row["er_start_ms"]:
            continue
        epoch_idx = int(row["epoch_index"])
        records.append((stim_times_s[epoch_idx] + row["er_start_ms"] / 1000,
                        (row["er_end_ms"] - row["er_start_ms"]) / 1000))
    return mne.Annotations(
        onset=[item[0] for item in records], duration=[item[1] for item in records],
        description=[label] * len(records), orig_time=raw.annotations.orig_time,
    )


er_auto_annotations = annotations_from_er_rows(er_df[er_df["er_found"]], er_auto_label)
old_er_names = {er_auto_label, "ER_candidate_auto", "ER_significant_auto", "ER_verified_manual"}
keep = np.asarray([str(label) not in old_er_names for label in raw.annotations.description])
raw.set_annotations(raw.annotations[keep] + er_auto_annotations)

# Production output contains stimuli and automatic ERs only.
output_mask = np.isin(np.asarray(raw.annotations.description, dtype=str), [stim_label, er_auto_label])
auto_output = raw.annotations[output_mask]
auto_output.save(er_auto_path, overwrite=True)
er_features_path = outputs_dir / "er_candidate_features.csv"
er_df.to_csv(er_features_path, index=False)

print("ER epochs:", len(er_df))
print("Morphology candidates:", int(er_df["candidate_found"].sum()))
print("Significant ER_auto:", int(er_df["er_found"].sum()))
print("Rejected after morphology:", int((er_df["candidate_found"] & ~er_df["er_found"]).sum()))
print("Detection outcomes:")
print(er_df["er_detection_source"].value_counts(dropna=False))
print("Auto annotation file:", er_auto_path)
print("Feature table:", er_features_path)


Overwriting existing file.
ER epochs: 561
Morphology candidates: 128
Significant ER_auto: 86
Rejected after morphology: 42
Detection outcomes:
er_detection_source
no_morphology_candidate      433
significant_er                86
candidate_not_significant     42
Name: count, dtype: int64
Auto annotation file: /Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/outputs/annotations/stimulus_er_mat_auto-annot.fif
Feature table: /Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/outputs/er_candidate_features.csv


/var/folders/4f/nvx50d4x7h5362c46mct3m3c0000gn/T/ipykernel_10718/3116948102.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  er_df[column] = er_df[column].fillna(False).astype(bool)


In [38]:
# 3. Visual check of production ER_auto annotations (inspection only)
# The browser receives a copy, so any accidental edits are discarded when it closes.

if len(er_auto_annotations):
    qc_start_s = max(0.0, float(er_auto_annotations.onset[0] - 0.1))
    print("Opening inspection-only ER_auto browser. Changes in this window are not saved.")
    raw_qc = raw.copy()
    raw_qc.plot(
        start=qc_start_s, duration=5.0, n_channels=len(raw_qc.ch_names),
        scalings=emg_browser_kwargs["scalings"],
        title="Production ER QC — ER_auto (inspection only)",
        block=True,
    )
    del raw_qc
else:
    print("No ER_auto annotations available for visual QC.")


Opening inspection-only ER_auto browser. Changes in this window are not saved.
Using pyopengl with version 3.1.10
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none


In [39]:
# 4. QC distributions for production ER_auto detections

er_qc_df = er_df[er_df["er_found"]].copy()
if len(er_qc_df):
    metric_specs = [
        ("er_start_ms", "Onset latency", "ms", er_morphology_limits["er_start_ms"]),
        ("er_peak_ms", "Peak latency", "ms", er_morphology_limits["er_peak_ms"]),
        ("er_full_duration_ms", "Full ER duration", "ms", er_morphology_limits["er_full_duration_ms"]),
        ("er_width_ms", "Negative-core width", "ms", er_morphology_limits["er_width_ms"]),
        ("er_p2p_uv", "Peak-to-peak amplitude", amplitude_unit, (None, er_amplitude_ceiling)),
        ("er_p2p_noise_ratio", "Peak-to-peak / baseline noise", "ratio", (er_p2p_snr_min, None)),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for ax, (column, title, unit, limits) in zip(axes.ravel(), metric_specs):
        values = pd.to_numeric(er_qc_df[column], errors="coerce").dropna()
        ax.hist(values, bins=30, color="tab:blue", alpha=0.75)
        ax.axvline(values.median(), color="black", linestyle="--",
                   label=f"median {values.median():.3g}")
        if limits[0] is not None:
            ax.axvline(limits[0], color="red", linestyle=":", linewidth=1)
        if limits[1] is not None and np.isfinite(limits[1]):
            ax.axvline(limits[1], color="red", linestyle=":", linewidth=1)
        ax.set(title=title, xlabel=unit, ylabel="ER_auto count")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f"Production ER_auto QC (n={len(er_qc_df)}) | {er_threshold_source}")
    fig.tight_layout()
    er_qc_figure_path = figures_dir / "er_auto_qc_distributions.png"
    fig.savefig(er_qc_figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    display(er_qc_df[[
        "er_start_ms", "er_peak_ms", "er_full_duration_ms", "er_width_ms",
        "er_p2p_uv", "er_p2p_noise_ratio",
    ]].describe())
    print("QC figure:", er_qc_figure_path)
else:
    print("No significant ER_auto detections; QC plots were not created.")


,er_start_ms,er_peak_ms,er_full_duration_ms,er_width_ms,er_p2p_uv,er_p2p_noise_ratio
count,86.000000,86.000000,86.000000,86.000000,86.000000,86.000000
mean,14.962209,17.206395,18.215116,6.267442,1.733023,13.384312
std,2.141164,2.025278,3.866744,1.249583,0.705272,5.403930
min,11.250000,13.250000,9.000000,3.500000,0.634687,5.017499
25%,13.250000,15.750000,15.562500,5.500000,1.157187,8.900935
50%,15.125000,17.625000,18.750000,6.250000,1.622812,12.557964
75%,16.500000,18.687500,21.937500,7.187500,2.185391,17.143951
max,19.750000,21.750000,24.750000,9.000000,3.509375,26.664349


QC figure: /Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/outputs/figures/er_auto_qc_distributions.png


In [40]:
# Параметры для поиска MR: финальная калибровка по 16 ручным MR
# Phase boundary is the midpoint between the positive peak and following trough.
# Bounds are 5th–95th percentiles of the final manually corrected reference.

mr_start_latency_window_ms = (223.51, 264.57)
mr_positive_peak_window_ms = (231.00, 276.56)
mr_phase_crossing_window_ms = (234.38, 285.88)
mr_negative_peak_window_ms = (238.00, 291.69)
mr_end_latency_window_ms = (249.13, 291.91)
mr_min_peak_separation_ms = 3.75
mr_max_peak_separation_ms = 22.06
mr_min_positive_width_ms = 5.79
mr_max_positive_width_ms = 26.70
mr_min_negative_width_ms = 1.18
mr_max_negative_width_ms = 29.30
mr_min_duration_ms = 21.55
mr_max_duration_ms = 41.71

mr_phase_amp_noise_k = 0.60  # compatibility; fixed-area detector does not use phase morphology
mr_p2p_noise_k = 2.0
mr_proposed_start_ms = float(mr_start_latency_window_ms[0])
mr_proposed_end_ms = float(mr_end_latency_window_ms[1])
mr_proposed_duration_ms = mr_proposed_end_ms - mr_proposed_start_ms
mr_min_peak_distance_ms = 3.0
lr_gap_after_mr_peak_ms = 0.0  # compatibility name; LR starts exactly at MR end

dt_ms = float(np.median(np.diff(times_ms)))
mr_min_peak_distance_samples = max(1, int(round(mr_min_peak_distance_ms / dt_ms)))

print("Final MR reference: 16 manually corrected spans")
print("Boundary rule: peak-to-trough midpoint; global baseline crossing is not required")


Final MR reference: 16 manually corrected spans
Boundary rule: peak-to-trough midpoint; global baseline crossing is not required


In [41]:
# Функции MR

def nearest_idx(t_ms):
    return idx_at(times_ms, t_ms)


def row_float(row, col, default=np.nan):
    return as_float(row.get(col, default), default)


def empty_mr(epoch_idx, reason, candidate_found=False, **candidate):
    row = {
        "epoch_index": int(epoch_idx), "mr_candidate_found": bool(candidate_found),
        "mr_found": False, "mr_detection_source": reason,
        "mr_search_start_ms": mr_start_latency_window_ms[0],
        "mr_search_end_ms": mr_end_latency_window_ms[1],
        "mr_start_ms": np.nan, "mr_peak_ms": np.nan,
        "mr_phase_crossing_ms": np.nan, "mr_negative_peak_ms": np.nan,
        "mr_end_ms": np.nan, "mr_peak_uv": np.nan, "mr_negative_peak_uv": np.nan,
        "mr_positive_amp_uv": np.nan, "mr_negative_amp_uv": np.nan,
        "mr_amp_uv": np.nan, "mr_p2p_uv": np.nan,
        "mr_positive_width_ms": np.nan, "mr_negative_width_ms": np.nan,
        "mr_duration_ms": np.nan, "mr_peak_separation_ms": np.nan,
        "mr_positive_noise_ratio": np.nan, "mr_negative_noise_ratio": np.nan,
        "mr_p2p_noise_ratio": np.nan, "mr_positive_prominence_noise_ratio": np.nan,
        "mr_negative_prominence_noise_ratio": np.nan, "mr_candidates_examined": 0,
        "er_amp_for_mr_uv": np.nan, "mr_er_ratio": np.nan,
        "lr_search_start_ms": np.nan,
    }
    row.update(candidate)
    return row


def mr_midpoint_bounds(y, pos_idx, neg_idx):
    level = 0.5 * (y[pos_idx] + y[neg_idx])
    start_limit = nearest_idx(max(mr_start_latency_window_ms[0], times_ms[pos_idx] - mr_max_positive_width_ms))
    end_limit = nearest_idx(min(mr_end_latency_window_ms[1], times_ms[neg_idx] + mr_max_negative_width_ms))
    before = np.where(y[start_limit:pos_idx + 1] <= level)[0]
    between = np.where(y[pos_idx:neg_idx + 1] <= level)[0]
    after = np.where(y[neg_idx:end_limit + 1] >= level)[0]
    if not len(before) or not len(between) or not len(after):
        return None
    start_idx = start_limit + int(before[-1])
    crossing_idx = pos_idx + int(between[0])
    end_idx = neg_idx + int(after[0])
    if not (start_idx < pos_idx < crossing_idx <= neg_idx < end_idx):
        return None
    return start_idx, crossing_idx, end_idx, float(level)


In [42]:
# Поиск MR внутри эпохи

def detect_mr(epoch_idx, row):
    if not bool(row.get("er_found", False)):
        return empty_mr(epoch_idx, "er_not_found")

    signal_uv = epochs_data_uv[epoch_idx]
    baseline_uv = row_float(row, "baseline_median_uv", 0.0)
    epoch_noise_uv = row_float(row, "baseline_noise_uv", np.nan)
    if not np.isfinite(epoch_noise_uv) or epoch_noise_uv <= 0:
        epoch_noise_uv = global_noise_uv
    effective_noise_uv = max(float(epoch_noise_uv), float(global_noise_uv))
    area_indices = np.where(
        (times_ms >= mr_proposed_start_ms) & (times_ms <= mr_proposed_end_ms)
    )[0]
    if len(area_indices) < 2:
        return empty_mr(epoch_idx, "bad_mr_area", lr_search_start_ms=mr_proposed_end_ms)
    area = signal_uv[area_indices] - baseline_uv
    peak_idx = int(area_indices[int(np.argmax(area))])
    trough_idx = int(area_indices[int(np.argmin(area))])
    p2p = float(np.ptp(area))
    p2p_noise_ratio = p2p / effective_noise_uv
    er_amp_uv = row_float(row, "er_p2p_uv", np.nan)
    found = bool(np.isfinite(p2p_noise_ratio) and p2p_noise_ratio >= mr_p2p_noise_k)
    return {
        **empty_mr(epoch_idx, "proposed_area_snr" if found else "proposed_area_below_snr"),
        "mr_candidate_found": True, "mr_found": found,
        "mr_detection_source": "proposed_area_snr" if found else "proposed_area_below_snr",
        "mr_start_ms": mr_proposed_start_ms, "mr_end_ms": mr_proposed_end_ms,
        "mr_peak_ms": float(times_ms[peak_idx]),
        "mr_negative_peak_ms": float(times_ms[trough_idx]),
        "mr_peak_uv": float(signal_uv[peak_idx]),
        "mr_negative_peak_uv": float(signal_uv[trough_idx]),
        "mr_amp_uv": p2p, "mr_p2p_uv": p2p,
        "mr_duration_ms": mr_proposed_duration_ms,
        "mr_p2p_noise_ratio": p2p_noise_ratio, "mr_candidates_examined": 1,
        "mr_effective_noise_uv": effective_noise_uv, "er_amp_for_mr_uv": er_amp_uv,
        "mr_er_ratio": p2p / er_amp_uv if np.isfinite(er_amp_uv) and er_amp_uv > 0 else np.nan,
        "lr_search_start_ms": mr_proposed_end_ms + lr_gap_after_mr_peak_ms,
    }

    # Legacy morphology implementation retained below for reference; unreachable.
    signal_uv = epochs_data_uv[epoch_idx]
    baseline_uv = row_float(row, "baseline_median_uv", 0.0)
    noise_uv = row_float(row, "baseline_noise_uv", 1.0)
    if not np.isfinite(noise_uv) or noise_uv <= 0:
        noise_uv = 1.0
    y = signal_uv - baseline_uv

    er_amp_uv = row_float(row, "er_p2p_uv")
    if not np.isfinite(er_amp_uv) or er_amp_uv <= 0:
        er_amp_uv = row_float(row, "er_depth_uv")
    if not np.isfinite(er_amp_uv) or er_amp_uv <= 0:
        return empty_mr(epoch_idx, "bad_er_amp")

    pos_indices = np.where(
        (times_ms >= mr_positive_peak_window_ms[0])
        & (times_ms <= mr_positive_peak_window_ms[1])
    )[0]
    neg_indices = np.where(
        (times_ms >= mr_negative_peak_window_ms[0])
        & (times_ms <= mr_negative_peak_window_ms[1])
    )[0]
    if not len(pos_indices) or not len(neg_indices):
        return empty_mr(epoch_idx, "bad_mr_window")

    pos_local, pos_props = find_peaks(
        y[pos_indices], prominence=0, distance=mr_min_peak_distance_samples,
    )
    neg_local, neg_props = find_peaks(
        -y[neg_indices], prominence=0, distance=mr_min_peak_distance_samples,
    )
    pos_peaks = pos_indices[pos_local]
    neg_peaks = neg_indices[neg_local]
    if not len(pos_peaks):
        return empty_mr(epoch_idx, "no_positive_peak")

    candidates = []
    for pos_order, pos_idx in enumerate(pos_peaks):
        for neg_order, neg_idx in enumerate(neg_peaks):
            separation_ms = float(times_ms[neg_idx] - times_ms[pos_idx])
            if separation_ms < mr_min_peak_separation_ms or separation_ms > mr_max_peak_separation_ms:
                continue
            bounds = mr_midpoint_bounds(y, int(pos_idx), int(neg_idx))
            if bounds is None:
                continue
            start_idx, crossing_idx, end_idx, phase_level = bounds
            positive_width_ms = float(times_ms[crossing_idx] - times_ms[start_idx])
            negative_width_ms = float(times_ms[end_idx] - times_ms[crossing_idx])
            duration_ms = float(times_ms[end_idx] - times_ms[start_idx])
            positive_amp = float(y[pos_idx] - phase_level)
            negative_amp = float(phase_level - y[neg_idx])
            p2p = float(y[pos_idx] - y[neg_idx])
            positive_prominence = float(pos_props["prominences"][pos_order])
            negative_prominence = float(neg_props["prominences"][neg_order])
            candidate = {
                "mr_start_ms": float(times_ms[start_idx]),
                "mr_peak_ms": float(times_ms[pos_idx]),
                "mr_phase_crossing_ms": float(times_ms[crossing_idx]),
                "mr_negative_peak_ms": float(times_ms[neg_idx]),
                "mr_end_ms": float(times_ms[end_idx]),
                "mr_peak_uv": float(signal_uv[pos_idx]),
                "mr_negative_peak_uv": float(signal_uv[neg_idx]),
                "mr_positive_amp_uv": positive_amp,
                "mr_negative_amp_uv": negative_amp,
                "mr_amp_uv": positive_amp,
                "mr_p2p_uv": p2p,
                "mr_positive_width_ms": positive_width_ms,
                "mr_negative_width_ms": negative_width_ms,
                "mr_duration_ms": duration_ms,
                "mr_peak_separation_ms": separation_ms,
                "mr_positive_noise_ratio": positive_amp / noise_uv,
                "mr_negative_noise_ratio": negative_amp / noise_uv,
                "mr_p2p_noise_ratio": p2p / noise_uv,
                "mr_positive_prominence_noise_ratio": positive_prominence / noise_uv,
                "mr_negative_prominence_noise_ratio": negative_prominence / noise_uv,
                "er_amp_for_mr_uv": er_amp_uv,
                "mr_er_ratio": p2p / er_amp_uv,
            }
            failures = []
            if not mr_start_latency_window_ms[0] <= candidate["mr_start_ms"] <= mr_start_latency_window_ms[1]:
                failures.append("start_latency")
            if not mr_positive_peak_window_ms[0] <= candidate["mr_peak_ms"] <= mr_positive_peak_window_ms[1]:
                failures.append("positive_peak_latency")
            if not mr_phase_crossing_window_ms[0] <= candidate["mr_phase_crossing_ms"] <= mr_phase_crossing_window_ms[1]:
                failures.append("phase_crossing_latency")
            if not mr_negative_peak_window_ms[0] <= candidate["mr_negative_peak_ms"] <= mr_negative_peak_window_ms[1]:
                failures.append("negative_peak_latency")
            if not mr_end_latency_window_ms[0] <= candidate["mr_end_ms"] <= mr_end_latency_window_ms[1]:
                failures.append("end_latency")
            if positive_amp < mr_phase_amp_noise_k * noise_uv:
                failures.append("positive_amplitude")
            if negative_amp < mr_phase_amp_noise_k * noise_uv:
                failures.append("negative_amplitude")
            if p2p < mr_p2p_noise_k * noise_uv:
                failures.append("p2p")
            if not mr_min_positive_width_ms <= positive_width_ms <= mr_max_positive_width_ms:
                failures.append("positive_width")
            if not mr_min_negative_width_ms <= negative_width_ms <= mr_max_negative_width_ms:
                failures.append("negative_width")
            if not mr_min_duration_ms <= duration_ms <= mr_max_duration_ms:
                failures.append("duration")
            candidate["failures"] = failures
            candidates.append(candidate)

    if not candidates:
        reason = "no_positive_negative_pair" if len(neg_peaks) else "no_negative_peak"
        return empty_mr(epoch_idx, reason)

    passing = [candidate for candidate in candidates if not candidate["failures"]]
    pool = passing if passing else candidates
    best = max(pool, key=lambda candidate: (candidate["mr_p2p_noise_ratio"], candidate["mr_duration_ms"]))
    candidate_fields = {key: value for key, value in best.items() if key != "failures"}
    candidate_fields["mr_candidates_examined"] = len(candidates)
    if not passing:
        reason = "below_" + "+".join(best["failures"])
        return empty_mr(epoch_idx, reason, candidate_found=True, **candidate_fields)

    candidate_fields["lr_search_start_ms"] = best["mr_end_ms"] + lr_gap_after_mr_peak_ms
    return {
        **empty_mr(epoch_idx, "biphasic_positive_negative", candidate_found=True),
        **candidate_fields,
        "mr_found": True,
        "mr_detection_source": "biphasic_positive_negative",
    }


In [43]:
# Поиск MR по всей записи

if "epoch_index" not in er_df.columns:
    er_df = er_df.reset_index().rename(columns={"index": "epoch_index"})

mr_rows = [
    detect_mr(int(row["epoch_index"]), row)
    if 0 <= int(row["epoch_index"]) < len(epochs_data_uv)
    else empty_mr(int(row["epoch_index"]), "epoch_out_of_range")
    for _, row in er_df.iterrows()
]
mr_df = pd.DataFrame(mr_rows)
mr_cols = [col for col in mr_df.columns if col != "epoch_index"]
er_df = er_df.drop(columns=[col for col in mr_cols if col in er_df.columns], errors="ignore").merge(
    mr_df, on="epoch_index", how="left",
)
er_df["mr_er_ratio"] = pd.to_numeric(er_df["mr_er_ratio"], errors="coerce")

# A large MR is now a valid target morphology. Keep compatibility flags for the
# downstream LR cells, but never reject an epoch solely because MR exceeds ER.
er_df["reject_mr_like"] = False
er_df["reject_high_mr"] = False


In [44]:
# Проверка MR: accepted waves, near misses, and metric distributions

print("Epochs:", len(er_df))
print("ER epochs searched:", int(er_df["er_found"].sum()))
print("MR candidates with measurable biphasic bounds:", int(er_df["mr_candidate_found"].sum()))
print("MR accepted:", int(er_df["mr_found"].sum()))
print("MR detection outcomes:")
print(er_df["mr_detection_source"].value_counts(dropna=False))

summary_cols = [
    "epoch_index", "mr_start_ms", "mr_peak_ms", "mr_phase_crossing_ms",
    "mr_negative_peak_ms", "mr_end_ms", "mr_positive_width_ms",
    "mr_negative_width_ms", "mr_duration_ms", "mr_p2p_noise_ratio",
    "mr_er_ratio", "lr_search_start_ms", "mr_detection_source",
]
display(er_df.loc[er_df["mr_candidate_found"], summary_cols].head(20))

accepted_qc = er_df[er_df["mr_found"]].nlargest(12, "mr_p2p_noise_ratio")
near_miss_qc = er_df[er_df["mr_candidate_found"] & ~er_df["mr_found"]].nlargest(12, "mr_p2p_noise_ratio")
qc_df = pd.concat([
    accepted_qc.assign(qc_group="accepted"),
    near_miss_qc.assign(qc_group="near miss"),
], ignore_index=True)

if len(qc_df):
    n_cols = 4
    n_rows = int(np.ceil(len(qc_df) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 2.7 * n_rows), sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row) in zip(axes, qc_df.iterrows()):
        epoch_idx = int(row["epoch_index"])
        signal = epochs_data_uv[epoch_idx]
        ax.plot(times_ms, signal, color="black", linewidth=0.8)
        ax.axvspan(mr_start_latency_window_ms[0], mr_end_latency_window_ms[1], color="gray", alpha=0.08)
        if np.isfinite(row["mr_start_ms"]):
            color = "tab:green" if row["mr_found"] else "tab:orange"
            ax.axvspan(row["mr_start_ms"], row["mr_end_ms"], color=color, alpha=0.18)
            ax.axvline(row["mr_phase_crossing_ms"], color="gray", linestyle=":", linewidth=0.8)
            ax.scatter(
                [row["mr_peak_ms"], row["mr_negative_peak_ms"]],
                [row["mr_peak_uv"], row["mr_negative_peak_uv"]], s=20,
                color=["tab:red", "tab:blue"],
            )
        if row["mr_found"]:
            ax.axvline(row["lr_search_start_ms"], color="purple", linestyle="--", linewidth=0.8)
        ax.set_xlim(120, 340)
        ax.set_title(
            f"e{epoch_idx} | {row['qc_group']} | p2p/n {row['mr_p2p_noise_ratio']:.1f}\n"
            f"{row['mr_detection_source']}", fontsize=8,
        )
        ax.grid(alpha=0.25)
    for ax in axes[len(qc_df):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("Нет MR-кандидатов для визуальной проверки")

mr_valid = er_df[er_df["mr_found"]].copy()
if len(mr_valid):
    metric_specs = [
        ("mr_peak_ms", "Positive peak latency", "ms", mr_positive_peak_window_ms),
        ("mr_negative_peak_ms", "Negative peak latency", "ms", mr_negative_peak_window_ms),
        ("mr_peak_separation_ms", "Peak separation", "ms", (mr_min_peak_separation_ms, mr_max_peak_separation_ms)),
        ("mr_positive_width_ms", "Positive phase width", "ms", (mr_min_positive_width_ms, mr_max_positive_width_ms)),
        ("mr_negative_width_ms", "Negative phase width", "ms", (mr_min_negative_width_ms, mr_max_negative_width_ms)),
        ("mr_duration_ms", "Total MR duration", "ms", (mr_min_duration_ms, mr_max_duration_ms)),
        ("mr_p2p_noise_ratio", "MR P2P / baseline noise", "ratio", (mr_p2p_noise_k, None)),
        ("mr_er_ratio", "MR P2P / ER amplitude", "ratio", (None, None)),
    ]
    fig, axes = plt.subplots(2, 4, figsize=(17, 7))
    for ax, (column, title, unit, limits) in zip(axes.ravel(), metric_specs):
        values = pd.to_numeric(mr_valid[column], errors="coerce").dropna()
        ax.hist(values, bins=20, color="tab:green", alpha=0.75)
        ax.axvline(values.median(), color="black", linestyle="--", label=f"median {values.median():.3g}")
        if limits[0] is not None:
            ax.axvline(limits[0], color="red", linestyle=":", linewidth=1)
        if limits[1] is not None:
            ax.axvline(limits[1], color="red", linestyle=":", linewidth=1)
        ax.set(title=title, xlabel=unit, ylabel="MR count")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f"Final midpoint-calibrated MR detections (n={len(mr_valid)})")
    fig.tight_layout()
    plt.show()

# Inspection-only view of SNR-qualified fixed MR_auto areas.
mr_label = "MR_auto"
mr_detected_before_review = er_df.query("mr_found == True").dropna(
    subset=["mr_start_ms", "mr_end_ms"],
).copy()
mr_review_records = []
for _, row in mr_detected_before_review.iterrows():
    if row["mr_end_ms"] <= row["mr_start_ms"]:
        continue
    epoch_idx = int(row["epoch_index"])
    onset_s = stim_times_s[epoch_idx] + row["mr_start_ms"] / 1000
    duration_s = (row["mr_end_ms"] - row["mr_start_ms"]) / 1000
    mr_review_records.append({
        "epoch_index": epoch_idx, "onset_s": onset_s,
        "end_s": onset_s + duration_s, "duration_s": duration_s,
    })
mr_review_source_df = pd.DataFrame(mr_review_records)
mr_auto_before_review = mne.Annotations(
    onset=mr_review_source_df["onset_s"].to_numpy() if len(mr_review_source_df) else [],
    duration=mr_review_source_df["duration_s"].to_numpy() if len(mr_review_source_df) else [],
    description=[mr_label] * len(mr_review_source_df),
    orig_time=raw.annotations.orig_time,
)

old_labels = np.asarray(raw.annotations.description, dtype=str)
raw.set_annotations(raw.annotations[old_labels != mr_label] + mr_auto_before_review)

open_mr_review_browser = os.environ.get("EMG_SKIP_INTERACTIVE_QC", "0") != "1"
if open_mr_review_browser and len(mr_auto_before_review):
    print("Opening inspection-only MR_auto browser; edits are discarded.")
    review_start_s = max(0.0, float(mr_auto_before_review.onset[0] - 0.1))
    raw_qc = raw.copy()
    raw_qc.plot(
        start=review_start_s, duration=5.0, n_channels=len(raw_qc.ch_names),
        scalings=emg_browser_kwargs["scalings"],
        title="Production MR QC — fixed MR_auto areas (inspection only)",
        block=True,
    )
    del raw_qc
else:
    print("Interactive MR review skipped; all automatic MR spans are retained.")

print("MR_auto fixed areas retained by SNR:", int(er_df["mr_found"].sum()))


Epochs: 561
ER epochs searched: 86
MR candidates with measurable biphasic bounds: 86
MR accepted: 86
MR detection outcomes:
mr_detection_source
er_not_found         475
proposed_area_snr     86
Name: count, dtype: int64


,epoch_index,mr_start_ms,mr_peak_ms,mr_phase_crossing_ms,mr_negative_peak_ms,mr_end_ms,mr_positive_width_ms,mr_negative_width_ms,mr_duration_ms,mr_p2p_noise_ratio,mr_er_ratio,lr_search_start_ms,mr_detection_source
22,22,223.51,290.25,NaN,229.50,291.91,NaN,NaN,68.4,2.553245,0.391442,291.91,proposed_area_snr
27,27,223.51,291.25,NaN,261.25,291.91,NaN,NaN,68.4,2.715771,0.541260,291.91,proposed_area_snr
32,32,223.51,291.75,NaN,223.75,291.91,NaN,NaN,68.4,2.070881,0.368340,291.91,proposed_area_snr
34,34,223.51,241.50,NaN,291.00,291.91,NaN,NaN,68.4,3.548183,0.441337,291.91,proposed_area_snr
36,36,223.51,233.50,NaN,291.75,291.91,NaN,NaN,68.4,3.780766,0.660795,291.91,proposed_area_snr
37,37,223.51,291.75,NaN,243.00,291.91,NaN,NaN,68.4,4.837727,0.480493,291.91,proposed_area_snr
38,38,223.51,272.50,NaN,247.50,291.91,NaN,NaN,68.4,5.460532,0.967934,291.91,proposed_area_snr
39,39,223.51,291.75,NaN,235.50,291.91,NaN,NaN,68.4,10.838623,1.738942,291.91,proposed_area_snr
43,43,223.51,224.25,NaN,274.25,291.91,NaN,NaN,68.4,6.153506,0.692519,291.91,proposed_area_snr
44,44,223.51,225.00,NaN,276.00,291.91,NaN,NaN,68.4,5.780980,0.665378,291.91,proposed_area_snr


Opening inspection-only MR_auto browser; edits are discarded.
Using pyopengl with version 3.1.10
Channels marked as bad:
none
MR_auto fixed areas retained by SNR: 86


In [45]:
# Save SNR-qualified fixed MR_auto annotations

mr_label = "MR_auto"
labels = np.asarray(raw.annotations.description, dtype=str)
mr_annotations = raw.annotations[labels == mr_label]

# Save the current automatic layers exactly as produced by the detectors.
save_labels = {stim_label, er_auto_label, mr_label}
save_mask = np.asarray([str(label) in save_labels for label in raw.annotations.description])
filtered_auto_annotations = raw.annotations[save_mask]
mr_annotation_path = annotations_dir / "stimulus_er_mr_mat_auto-annot.fif"
mr_annotation_tmp_path = annotations_dir / ".stimulus_er_mr_mat_auto.tmp-annot.fif"
filtered_auto_annotations.save(mr_annotation_tmp_path, overwrite=True)
roundtrip = mne.read_annotations(mr_annotation_tmp_path)
expected_counts = pd.Series(filtered_auto_annotations.description, dtype=str).value_counts().to_dict()
actual_counts = pd.Series(roundtrip.description, dtype=str).value_counts().to_dict()
if len(roundtrip) != len(filtered_auto_annotations) or actual_counts != expected_counts:
    raise RuntimeError("Filtered MR annotation round-trip validation failed")
os.replace(mr_annotation_tmp_path, mr_annotation_path)

print("Fixed MR_auto annotations saved:", len(mr_annotations))
print("Saved annotation counts:", actual_counts)
print("Output:", mr_annotation_path)


Fixed MR_auto annotations saved: 86
Saved annotation counts: {'Stimulus_Auto': 562, 'ER_auto': 86, 'MR_auto': 86}
Output: /Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/outputs/annotations/stimulus_er_mr_mat_auto-annot.fif


In [46]:
# Прямой поиск LR: 1–3 похожих положительных пика после MR

need("er_df", "epochs_data_uv", "times_ms", "find_peaks")

# 5th–95th percentile calibration from 134 manually retained LR_auto spans.
lr_peak_noise_k = 2.5
lr_prominence_noise_k = 2.0
lr_min_peak_width_ms = 8.0
lr_max_peak_width_ms = 26.3
lr_min_duration_ms = 10.0
lr_max_duration_ms = 42.2
lr_min_peak_distance_ms = 10.0
# Peaks retained in the same epoch must be similar to the strongest peak.
lr_peak_similarity_tolerance = 0.25  # ±25% amplitude and half-height width
lr_max_peaks_per_epoch = 3

dt_ms = float(np.median(np.diff(times_ms)))
lr_min_peak_distance_samples = max(1, int(round(lr_min_peak_distance_ms / dt_ms)))
baseline_by_epoch = er_df.set_index("epoch_index")["baseline_median_uv"].reindex(
    np.arange(len(epochs_data_uv)),
).to_numpy()
baseline_by_epoch = np.where(
    np.isfinite(baseline_by_epoch), baseline_by_epoch, np.nanmedian(baseline_by_epoch),
)
epochs_centered_uv = epochs_data_uv - baseline_by_epoch[:, None]

lr_fixed_start_ms = mr_proposed_end_ms + lr_gap_after_mr_peak_ms
lr_rejection_counts = {"amplitude": 0, "prominence": 0, "peak_width": 0, "duration": 0}


def lr_window(row):
    isi_ms = as_float(row.get("isi_ms", np.nan))
    next_stim_ms = isi_ms if np.isfinite(isi_ms) else times_ms[-1]
    end_ms = min(float(next_stim_ms), float(times_ms[-1]))
    return float(lr_fixed_start_ms), float(end_ms), "fixed_max_mr_end"


def half_height_bounds(signal, peak_idx, left_limit, right_limit):
    half_height = 0.5 * signal[peak_idx]
    left = peak_idx
    while left > left_limit and signal[left - 1] >= half_height:
        left -= 1
    right = peak_idx
    while right < right_limit and signal[right + 1] >= half_height:
        right += 1
    return left, right


def detect_lr_positive_peaks(epoch_idx, signal, row):
    if not bool(row.get("er_found", False)):
        return []
    epoch_noise = as_float(row.get("baseline_noise_uv", np.nan))
    if not np.isfinite(epoch_noise) or epoch_noise <= 0:
        epoch_noise = global_noise_uv
    effective_noise = max(float(epoch_noise), float(global_noise_uv))
    start_ms, end_ms, start_source = lr_window(row)
    if not np.isfinite([start_ms, end_ms]).all() or end_ms <= start_ms:
        return []
    i0, i1 = idx_at(times_ms, start_ms), idx_at(times_ms, end_ms)
    if i1 <= i0 + 2:
        return []

    segment = signal[i0:i1 + 1]
    local_peaks, properties = find_peaks(
        segment, prominence=0, distance=lr_min_peak_distance_samples,
    )
    candidates = []
    er_p2p = as_float(row.get("er_p2p_uv", np.nan))
    for order, local_peak in enumerate(local_peaks):
        peak_idx = i0 + int(local_peak)
        amplitude = float(segment[local_peak])
        prominence = float(properties["prominences"][order])
        if amplitude < lr_peak_noise_k * effective_noise:
            lr_rejection_counts["amplitude"] += 1
            continue
        if prominence < lr_prominence_noise_k * effective_noise:
            lr_rejection_counts["prominence"] += 1
            continue
        peak_width_ms = float(peak_widths(segment, [local_peak], rel_height=0.5)[0][0] * dt_ms)
        if not lr_min_peak_width_ms <= peak_width_ms <= lr_max_peak_width_ms:
            lr_rejection_counts["peak_width"] += 1
            continue
        left, right = half_height_bounds(signal, peak_idx, i0, i1)
        width_ms = float(times_ms[right] - times_ms[left])
        if right <= left or not lr_min_duration_ms <= width_ms <= lr_max_duration_ms:
            lr_rejection_counts["duration"] += 1
            continue
        candidates.append({
            "epoch_index": int(epoch_idx),
            "stim_time_s": as_float(row.get("stim_time_s", np.nan)),
            "method": "positive_peak_direct", "candidate_rank": 0,
            "lr_window_start_ms": start_ms, "lr_window_end_ms": end_ms,
            "lr_window_start_source": start_source,
            "lr_wave_start_ms": float(times_ms[left]),
            "lr_wave_end_ms": float(times_ms[right]),
            "lr_duration_ms": width_ms,
            "lr_peak_width_ms": peak_width_ms,
            "lr_start_ms": float(times_ms[left]),
            "lr_peak_ms": float(times_ms[peak_idx]),
            "lr_end_ms": float(times_ms[right]),
            "lr_p2p_uv": amplitude, "lr_peak_uv": amplitude,
            "lr_prominence_uv": prominence,
            "lr_ratio": amplitude / er_p2p if np.isfinite(er_p2p) and er_p2p > 0 else np.nan,
            "lr_noise_ratio": amplitude / effective_noise,
            "lr_peak_noise_ratio": amplitude / effective_noise,
            "lr_prominence_noise_ratio": prominence / effective_noise,
            "lr_polarity": "positive", "baseline_noise_uv": epoch_noise,
            "lr_effective_noise_uv": effective_noise,
            "er_p2p_uv": er_p2p,
        })
    if not candidates:
        return []

    strongest = max(candidates, key=lambda candidate: candidate["lr_peak_uv"])
    amplitude_scale = max(strongest["lr_peak_uv"], np.finfo(float).eps)
    width_scale = max(strongest["lr_peak_width_ms"], dt_ms)
    similar = [
        candidate for candidate in candidates
        if abs(candidate["lr_peak_uv"] - strongest["lr_peak_uv"]) / amplitude_scale
        <= lr_peak_similarity_tolerance
        and abs(candidate["lr_peak_width_ms"] - strongest["lr_peak_width_ms"]) / width_scale
        <= lr_peak_similarity_tolerance
    ]
    kept = sorted(similar, key=lambda candidate: candidate["lr_peak_ms"])[:lr_max_peaks_per_epoch]
    for rank, candidate in enumerate(kept, start=1):
        candidate["candidate_rank"] = rank
    return kept


lr_candidate_cols = [
    "epoch_index", "stim_time_s", "method", "candidate_rank",
    "lr_window_start_ms", "lr_window_end_ms", "lr_window_start_source",
    "lr_wave_start_ms", "lr_wave_end_ms", "lr_duration_ms", "lr_peak_width_ms",
    "lr_start_ms", "lr_peak_ms", "lr_end_ms", "lr_p2p_uv", "lr_peak_uv",
    "lr_prominence_uv", "lr_ratio", "lr_noise_ratio", "lr_peak_noise_ratio",
    "lr_prominence_noise_ratio", "lr_polarity", "baseline_noise_uv",
    "lr_effective_noise_uv", "er_p2p_uv",
]
raw_lr_rows = [
    candidate
    for _, row in er_df.iterrows()
    for epoch_idx in [int(row["epoch_index"])]
    if 0 <= epoch_idx < len(epochs_centered_uv)
    for candidate in detect_lr_positive_peaks(epoch_idx, epochs_centered_uv[epoch_idx], row)
]
lr_candidates_raw_df = pd.DataFrame(raw_lr_rows, columns=lr_candidate_cols)
lr_candidate_sensitive = set(lr_candidates_raw_df["epoch_index"].astype(int)) if len(lr_candidates_raw_df) else set()

print("Direct positive-peak LR detection")
print("ER epochs searched:", int(er_df["er_found"].sum()))
print("Fixed LR search start, ms:", lr_fixed_start_ms)
print("LR peak rows:", len(lr_candidates_raw_df))
print("Epochs with LR peaks:", len(lr_candidate_sensitive))
print("Rejected by first failed LR filter:", lr_rejection_counts)
if len(lr_candidates_raw_df):
    display(lr_candidates_raw_df[[
        "epoch_index", "candidate_rank", "lr_window_start_source", "lr_peak_ms",
        "lr_duration_ms", "lr_peak_width_ms", "lr_peak_uv",
        "lr_noise_ratio", "lr_prominence_noise_ratio",
    ]].head(30))


Direct positive-peak LR detection
ER epochs searched: 86
Fixed LR search start, ms: 291.91
LR peak rows: 39
Epochs with LR peaks: 32
Rejected by first failed LR filter: {'amplitude': 682, 'prominence': 96, 'peak_width': 16, 'duration': 18}


,epoch_index,candidate_rank,lr_window_start_source,lr_peak_ms,lr_duration_ms,lr_peak_width_ms,lr_peak_uv,lr_noise_ratio,lr_prominence_noise_ratio
0,34,1,fixed_max_mr_end,433.00,11.25,14.164027,0.316563,2.617851,3.393128
1,55,1,fixed_max_mr_end,379.00,19.00,18.893795,1.152188,7.970665,7.836632
2,188,1,fixed_max_mr_end,474.75,13.75,13.705164,2.604063,21.534603,20.425957
3,231,1,fixed_max_mr_end,362.25,14.75,20.834606,0.433125,3.280155,4.404306
4,291,1,fixed_max_mr_end,393.25,25.25,16.850695,0.641250,5.302893,4.406156
5,297,1,fixed_max_mr_end,314.00,12.25,9.033593,0.361563,2.989984,2.069989
6,299,1,fixed_max_mr_end,314.25,14.25,8.833958,0.432187,3.574026,2.015719
7,299,2,fixed_max_mr_end,334.25,12.00,9.278125,0.360625,2.982231,2.188865
8,300,1,fixed_max_mr_end,314.50,12.00,9.623106,0.368750,3.049422,2.276729
9,312,1,fixed_max_mr_end,315.00,12.00,8.493667,0.588437,4.006517,2.612854


In [60]:
# Проверка первичного LR

need("lr_candidates_raw_df", "epochs_centered_uv", "times_ms")

plot_meta = lr_candidates_raw_df.sort_values(["lr_ratio", "lr_noise_ratio"], ascending=False).head(30).copy()

if len(plot_meta) == 0:
    print("Нет первичных LR-кандидатов")
else:
    n_cols = 5
    n_rows = int(np.ceil(len(plot_meta) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 2.8 * n_rows), sharex=True)
    axes = np.ravel(axes)

    er_lookup = er_df.set_index("epoch_index")
    for ax, (_, cand) in zip(axes, plot_meta.iterrows()):
        epoch_idx = int(cand["epoch_index"])
        row = er_lookup.loc[epoch_idx]
        ax.plot(times_ms, epochs_centered_uv[epoch_idx], linewidth=0.9)
        ax.axvline(0, linestyle=":", linewidth=0.8)
        ax.axvspan(cand["lr_window_start_ms"], cand["lr_window_end_ms"], alpha=0.08)
        ax.axvspan(cand["lr_start_ms"], cand["lr_end_ms"], alpha=0.25)
        ax.axvline(cand["lr_peak_ms"], linewidth=0.8)
        if pd.notna(row.get("er_start_ms", np.nan)) and pd.notna(row.get("er_end_ms", np.nan)):
            ax.axvspan(row["er_start_ms"], row["er_end_ms"], alpha=0.10)
        if pd.notna(row.get("mr_start_ms", np.nan)) and pd.notna(row.get("mr_end_ms", np.nan)):
            ax.axvspan(row["mr_start_ms"], row["mr_end_ms"], alpha=0.12)
        ax.set_title(f"epoch {epoch_idx} | dur {cand['lr_duration_ms']:.2f} | p2p {cand['lr_p2p_uv']:.0f}", fontsize=8)
        ax.grid(alpha=0.25)

    for ax in axes[len(plot_meta):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [58]:
# Consolidate direct LR peaks for QC, review, and annotation

need("lr_candidates_raw_df", "epochs_centered_uv", "er_df")

lr_candidates_residual_df = lr_candidates_raw_df.copy()  # compatibility alias
lr_keys = [
    "wave_start_ms", "wave_end_ms", "duration_ms", "start_ms", "peak_ms", "end_ms",
    "p2p_uv", "peak_uv", "prominence_uv", "peak_width_ms", "ratio", "noise_ratio",
    "peak_noise_ratio", "prominence_noise_ratio", "polarity",
    "method", "confidence", "confidence_score",
]


def lr_slot(candidate=None):
    if candidate is None:
        return dict(zip(lr_keys, [np.nan] * 14 + ["none", "none", "none", 0]))
    return {
        "wave_start_ms": as_float(candidate["lr_wave_start_ms"]),
        "wave_end_ms": as_float(candidate["lr_wave_end_ms"]),
        "duration_ms": as_float(candidate["lr_duration_ms"]),
        "start_ms": as_float(candidate["lr_start_ms"]),
        "peak_ms": as_float(candidate["lr_peak_ms"]),
        "end_ms": as_float(candidate["lr_end_ms"]),
        "p2p_uv": as_float(candidate["lr_p2p_uv"]),
        "peak_uv": as_float(candidate["lr_peak_uv"]),
        "prominence_uv": as_float(candidate["lr_prominence_uv"]),
        "peak_width_ms": as_float(candidate["lr_peak_width_ms"]),
        "ratio": as_float(candidate["lr_ratio"]),
        "noise_ratio": as_float(candidate["lr_noise_ratio"]),
        "peak_noise_ratio": as_float(candidate["lr_peak_noise_ratio"]),
        "prominence_noise_ratio": as_float(candidate["lr_prominence_noise_ratio"]),
        "polarity": candidate["lr_polarity"], "method": candidate["method"],
        "confidence": "review", "confidence_score": 1,
    }


records = []
for _, er_row in er_df.iterrows():
    epoch_idx = int(er_row["epoch_index"])
    epoch_lr = lr_candidates_raw_df.query("epoch_index == @epoch_idx").sort_values("candidate_rank")
    base = {
        "epoch_index": epoch_idx, "stim_time_s": as_float(er_row.get("stim_time_s", np.nan)),
        "isi_ms": as_float(er_row.get("isi_ms", np.nan)),
        "baseline_noise_uv": as_float(er_row.get("baseline_noise_uv", np.nan)),
        "er_peak_ms": as_float(er_row.get("er_peak_ms", np.nan)),
        "er_start_ms": as_float(er_row.get("er_start_ms", np.nan)),
        "er_end_ms": as_float(er_row.get("er_end_ms", np.nan)),
        "er_p2p_uv": as_float(er_row.get("er_p2p_uv", np.nan)),
        "mr_found": bool(er_row.get("mr_found", False)),
        "mr_start_ms": as_float(er_row.get("mr_start_ms", np.nan)),
        "mr_peak_ms": as_float(er_row.get("mr_peak_ms", np.nan)),
        "mr_end_ms": as_float(er_row.get("mr_end_ms", np.nan)),
        "mr_er_ratio": as_float(er_row.get("mr_er_ratio", np.nan)),
        "reject_mr_like": False, "n_lr": int(len(epoch_lr)),
        "n_lr_threshold": int(len(epoch_lr)), "n_lr_residual": int(len(epoch_lr)),
        "epoch_status": "lr_found" if len(epoch_lr) else "no_lr",
        "confidence_score": 1 if len(epoch_lr) else 0,
    }
    for slot in [1, 2, 3]:
        values = lr_slot(epoch_lr.iloc[slot - 1] if slot <= len(epoch_lr) else None)
        base.update({f"lr{slot}_{key}": value for key, value in values.items()})
    records.append(base)
lr_candidates_df = pd.DataFrame(records)


def lr_auc_rms(epoch_idx, start_ms, end_ms):
    if not np.isfinite([start_ms, end_ms]).all() or end_ms <= start_ms:
        return np.nan, np.nan
    mask = (times_ms >= start_ms) & (times_ms <= end_ms)
    if mask.sum() < 2:
        return np.nan, np.nan
    segment = epochs_centered_uv[epoch_idx, mask]
    return (
        float(np.trapezoid(np.abs(segment), times_ms[mask])),
        float(np.sqrt(np.mean(segment ** 2))),
    )


for slot in [1, 2, 3]:
    auc_values, rms_values = [], []
    for _, row in lr_candidates_df.iterrows():
        auc, rms = lr_auc_rms(
            int(row["epoch_index"]),
            as_float(row.get(f"lr{slot}_wave_start_ms", np.nan)),
            as_float(row.get(f"lr{slot}_wave_end_ms", np.nan)),
        )
        auc_values.append(auc); rms_values.append(rms)
    lr_candidates_df[f"lr{slot}_auc_abs_uv_ms"] = auc_values
    lr_candidates_df[f"lr{slot}_rms_uv"] = rms_values

print("Direct LR candidate table")
print("Peak rows:", len(lr_candidates_raw_df))
print("Epochs with LR:", int((lr_candidates_df["n_lr"] > 0).sum()))


Direct LR candidate table
Peak rows: 39
Epochs with LR: 32


In [56]:
# Проверка прямых LR-кандидатов

need("lr_candidates_df", "epochs_centered_uv", "times_ms")
plot_meta = lr_candidates_df.loc[lr_candidates_df["n_lr"] > 0].head(30).copy()
if len(plot_meta) == 0:
    print("Нет LR-кандидатов")
else:
    n_cols = 5
    n_rows = int(np.ceil(len(plot_meta) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 2.8 * n_rows), sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row) in zip(axes, plot_meta.iterrows()):
        epoch_idx = int(row["epoch_index"])
        ax.plot(times_ms, epochs_centered_uv[epoch_idx], linewidth=0.9)
        if np.isfinite(row["mr_end_ms"]):
            ax.axvline(row["mr_end_ms"], color="orange", linestyle="--", linewidth=0.9)
        else:
            ax.axvline(lr_fixed_start_ms, color="gray", linestyle="--", linewidth=0.9)
        for slot in [1, 2, 3]:
            start_ms = row.get(f"lr{slot}_start_ms", np.nan)
            end_ms = row.get(f"lr{slot}_end_ms", np.nan)
            peak_ms = row.get(f"lr{slot}_peak_ms", np.nan)
            if np.isfinite([start_ms, end_ms]).all():
                ax.axvspan(start_ms, end_ms, color="tab:blue", alpha=0.20)
            if np.isfinite(peak_ms):
                ax.axvline(peak_ms, color="tab:blue", linewidth=0.7)
        ax.set_title(f"epoch {epoch_idx} | peaks {int(row['n_lr'])}", fontsize=8)
        ax.grid(alpha=0.25)
    for ax in axes[len(plot_meta):]:
        ax.axis("off")
    fig.tight_layout(); plt.show()


In [57]:
from pathlib import Path
annotations_dir = Path("annotations")
annotations_dir.mkdir(parents=True, exist_ok=True)
output_path = annotations_dir / "manual_er_mr_lr-annot.fif"
raw.annotations.save(output_path, overwrite=True)
print(output_path.resolve())

Overwriting existing file.
/Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/notebooks/annotations/manual_er_mr_lr-annot.fif


In [61]:
# Распределения LR

need("lr_candidates_df")

lr_parts = []
for k in [1, 2, 3]:
    cols = {
        f"lr{k}_peak_ms": "latency_ms", f"lr{k}_duration_ms": "duration_ms", f"lr{k}_p2p_uv": "p2p_uv",
        f"lr{k}_peak_uv": "peak_uv", f"lr{k}_prominence_uv": "prominence_uv",
        f"lr{k}_peak_width_ms": "peak_width_ms", f"lr{k}_noise_ratio": "noise_ratio",
        f"lr{k}_peak_noise_ratio": "peak_noise_ratio",
        f"lr{k}_prominence_noise_ratio": "prominence_noise_ratio",
        f"lr{k}_auc_abs_uv_ms": "auc_abs_uv_ms", f"lr{k}_rms_uv": "rms_uv",
    }
    part = lr_candidates_df.loc[(lr_candidates_df["reject_mr_like"] == False) & lr_candidates_df[f"lr{k}_peak_ms"].notna(), ["epoch_index", *cols.keys()]].rename(columns=cols)
    part["lr_n"] = k
    lr_parts.append(part)

lr_distribution_df = pd.concat(lr_parts, ignore_index=True) if lr_parts else pd.DataFrame()
duration_values = lr_distribution_df["duration_ms"].dropna().to_numpy() if len(lr_distribution_df) else np.array([])
bin_step_ms = max(0.25, float(dt_ms))

if len(duration_values):
    duration_bins = np.arange(0, np.nanmax(duration_values) + 2 * bin_step_ms, bin_step_ms)
    duration_counts, duration_edges = np.histogram(duration_values, bins=duration_bins)
    mode_i = int(np.argmax(duration_counts))
    lr_duration_mode_ms = float((duration_edges[mode_i] + duration_edges[mode_i + 1]) / 2)
    lr_duration_mode_window_ms = (float(duration_edges[mode_i]), float(duration_edges[mode_i + 1]))
else:
    lr_duration_mode_ms = np.nan
    lr_duration_mode_window_ms = (np.nan, np.nan)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
axes = axes.ravel()
plot_specs = [
    ("latency_ms", "LR latency", "ms"),
    ("duration_ms", "LR duration", "ms"),
    ("p2p_uv", "LR p2p", amplitude_unit),
    ("auc_abs_uv_ms", "LR absolute AUC", f"{amplitude_unit}*ms"),
    ("rms_uv", "LR RMS", amplitude_unit),
    ("peak_width_ms", "LR half-prominence width", "ms"),
    ("noise_ratio", "LR amplitude / effective noise", "ratio"),
    ("prominence_noise_ratio", "LR prominence / effective noise", "ratio"),
]

for ax, (col, title, xlabel) in zip(axes, plot_specs):
    ax.hist(lr_distribution_df[col].dropna() if len(lr_distribution_df) else [], bins=30, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(alpha=0.25)

if np.isfinite(lr_duration_mode_ms):
    axes[1].axvline(lr_duration_mode_ms, color="red", linewidth=1.5)
for ax, threshold in [
    (axes[1], lr_min_duration_ms), (axes[1], lr_max_duration_ms),
    (axes[5], lr_min_peak_width_ms), (axes[5], lr_max_peak_width_ms),
    (axes[6], lr_peak_noise_k), (axes[7], lr_prominence_noise_k),
]:
    ax.axvline(threshold, color="red", linestyle=":", linewidth=1)

plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "n_lr": [len(lr_distribution_df)], "duration_mode_ms": [lr_duration_mode_ms],
    "duration_mode_left_ms": [lr_duration_mode_window_ms[0]], "duration_mode_right_ms": [lr_duration_mode_window_ms[1]],
    "duration_median_ms": [np.nanmedian(duration_values) if len(duration_values) else np.nan],
    "duration_q25_ms": [np.nanpercentile(duration_values, 25) if len(duration_values) else np.nan],
    "duration_q75_ms": [np.nanpercentile(duration_values, 75) if len(duration_values) else np.nan],
    "auc_abs_median_uv_ms": [np.nanmedian(lr_distribution_df["auc_abs_uv_ms"]) if len(lr_distribution_df) else np.nan],
    "rms_median_uv": [np.nanmedian(lr_distribution_df["rms_uv"]) if len(lr_distribution_df) else np.nan],
}))

lr_distribution_df.sort_values("duration_ms").head()

# Interactive LR filtering: delete false full-span LR_auto annotations.
lr_review_label = "LR_auto"
lr_review_records = []
for row_index, row in lr_candidates_df.iterrows():
    epoch_idx = int(row["epoch_index"])
    for slot in [1, 2, 3]:
        start_ms = as_float(row.get(f"lr{slot}_wave_start_ms", np.nan))
        end_ms = as_float(row.get(f"lr{slot}_wave_end_ms", np.nan))
        if not np.isfinite([start_ms, end_ms]).all() or end_ms <= start_ms:
            continue
        onset_s = stim_times_s[epoch_idx] + start_ms / 1000
        lr_review_records.append({
            "row_index": int(row_index), "epoch_index": epoch_idx, "slot": slot,
            "onset_s": onset_s, "end_s": stim_times_s[epoch_idx] + end_ms / 1000,
            "duration_s": (end_ms - start_ms) / 1000,
        })
lr_review_source_df = pd.DataFrame(lr_review_records)
lr_auto_before_review = mne.Annotations(
    onset=lr_review_source_df["onset_s"].to_numpy() if len(lr_review_source_df) else [],
    duration=lr_review_source_df["duration_s"].to_numpy() if len(lr_review_source_df) else [],
    description=[lr_review_label] * len(lr_review_source_df), orig_time=raw.annotations.orig_time,
)
old_labels = np.asarray(raw.annotations.description, dtype=str)
raw.set_annotations(raw.annotations[old_labels != lr_review_label] + lr_auto_before_review)

open_lr_review_browser = os.environ.get("EMG_SKIP_INTERACTIVE_QC", "0") != "1"
if open_lr_review_browser and len(lr_auto_before_review):
    print("LR review: press A, delete false LR_auto spans, then close the window.")
    print("Delete spans only; do not rename or resize them.")
    raw.plot(
        start=max(0.0, float(lr_auto_before_review.onset[0] - 0.1)), duration=5.0,
        n_channels=len(raw.ch_names), scalings=emg_browser_kwargs["scalings"],
        title="LR manual filtering — delete false LR_auto spans, then close", block=True,
    )
else:
    print("Interactive LR review skipped; all permissive LR candidates are retained.")

labels_after_lr_review = np.asarray(raw.annotations.description, dtype=str)
lr_reviewed_annotations = raw.annotations[labels_after_lr_review == lr_review_label]
retained_keys = set()
for _, original in lr_review_source_df.iterrows():
    overlaps = np.minimum(original["end_s"], lr_reviewed_annotations.onset + lr_reviewed_annotations.duration) - np.maximum(
        original["onset_s"], lr_reviewed_annotations.onset,
    )
    if len(overlaps) and np.max(overlaps) > 0:
        retained_keys.add((int(original["row_index"]), int(original["slot"])))

lr_value_fields = [
    "wave_start_ms", "wave_end_ms", "duration_ms", "start_ms", "peak_ms", "end_ms",
    "p2p_uv", "peak_uv", "prominence_uv", "peak_width_ms", "ratio", "noise_ratio",
    "peak_noise_ratio", "prominence_noise_ratio",
    "auc_abs_uv_ms", "rms_uv",
]
for row_index, row in lr_candidates_df.iterrows():
    retained_slots = []
    for slot in [1, 2, 3]:
        if (int(row_index), slot) in retained_keys:
            retained_slots.append(slot)
            continue
        for field in lr_value_fields:
            column = f"lr{slot}_{field}"
            if column in lr_candidates_df:
                lr_candidates_df.loc[row_index, column] = np.nan
        for field in ["polarity", "method", "confidence"]:
            column = f"lr{slot}_{field}"
            if column in lr_candidates_df:
                lr_candidates_df.loc[row_index, column] = "none"
        score_col = f"lr{slot}_confidence_score"
        if score_col in lr_candidates_df:
            lr_candidates_df.loc[row_index, score_col] = 0
    lr_candidates_df.loc[row_index, "n_lr"] = len(retained_slots)
    lr_candidates_df.loc[row_index, "epoch_status"] = "lr_found" if retained_slots else "no_lr"
    lr_candidates_df.loc[row_index, "confidence_score"] = max(
        int(lr_candidates_df.loc[row_index, "lr1_confidence_score"]),
        int(lr_candidates_df.loc[row_index, "lr2_confidence_score"]),
        int(lr_candidates_df.loc[row_index, "lr3_confidence_score"]),
    )

print("LR_auto before review:", len(lr_review_source_df))
print("LR_auto retained after review:", len(retained_keys))
print("LR_auto manually removed:", len(lr_review_source_df) - len(retained_keys))


,n_lr,duration_mode_ms,duration_mode_left_ms,duration_mode_right_ms,duration_median_ms,duration_q25_ms,duration_q75_ms,auc_abs_median_uv_ms,rms_median_uv
0,39,12.125,12.0,12.25,12.75,12.0,14.5,4.822578,0.345746


LR review: press A, delete false LR_auto spans, then close the window.
Delete spans only; do not rename or resize them.
Using pyopengl with version 3.1.10
Channels marked as bad:
none
LR_auto before review: 39
LR_auto retained after review: 39
LR_auto manually removed: 0


In [51]:
# Save manually filtered Stimulus + ER + MR + LR automatic annotations

lr_review_label = "LR_auto"
save_labels = {stim_label, er_auto_label, "MR_auto", lr_review_label}
save_mask = np.asarray([str(label) in save_labels for label in raw.annotations.description])
filtered_auto_annotations = raw.annotations[save_mask]
lr_annotation_path = annotations_dir / "stimulus_er_mr_lr_mat_auto-annot.fif"
lr_annotation_tmp_path = annotations_dir / ".stimulus_er_mr_lr_mat_auto.tmp-annot.fif"
filtered_auto_annotations.save(lr_annotation_tmp_path, overwrite=True)
roundtrip = mne.read_annotations(lr_annotation_tmp_path)
expected_counts = pd.Series(filtered_auto_annotations.description, dtype=str).value_counts().to_dict()
actual_counts = pd.Series(roundtrip.description, dtype=str).value_counts().to_dict()
if len(roundtrip) != len(filtered_auto_annotations) or actual_counts != expected_counts:
    raise RuntimeError("Filtered LR annotation round-trip validation failed")
os.replace(lr_annotation_tmp_path, lr_annotation_path)

print("Filtered LR_auto annotations saved:", actual_counts.get(lr_review_label, 0))
print("Saved annotation counts:", actual_counts)
print("Output:", lr_annotation_path)


Filtered LR_auto annotations saved: 21
Saved annotation counts: {'Stimulus_Auto': 562, 'MR_auto': 53, 'ER_auto': 50, 'LR_auto': 21}
Output: /Users/sofiaefimochkina/Documents/lab/emg-lr-segmentation/outputs/annotations/stimulus_er_mr_lr_mat_auto-annot.fif


In [52]:
# Финальные графики LR

need("lr_candidates_df", "epochs_data_uv", "times_ms")

n_plots = 10  # сколько эпох показать
min_confidence_score_to_plot = 1  # минимальный балл LR для графиков
show_reject_mr_like = True  # показывать эпохи с высоким MR

plot_meta = lr_candidates_df.copy()
reject_col = plot_meta["reject_mr_like"] if "reject_mr_like" in plot_meta else plot_meta.get("reject_high_mr", pd.Series(False, index=plot_meta.index))
plot_meta["reject_mr_like"] = reject_col.fillna(False).astype(bool)
plot_meta["has_ER"] = plot_meta["er_start_ms"].notna() & plot_meta["er_end_ms"].notna() & (plot_meta["er_p2p_uv"] > 0)
mr_found = plot_meta["mr_found"].fillna(False).astype(bool) if "mr_found" in plot_meta else pd.Series(True, index=plot_meta.index)
plot_meta["has_MR"] = mr_found & plot_meta["mr_start_ms"].notna() & plot_meta["mr_end_ms"].notna()
plot_meta["has_LR"] = (plot_meta["n_lr"] > 0) & (plot_meta["lr1_peak_ms"].notna() | plot_meta["lr2_peak_ms"].notna() | plot_meta["lr3_peak_ms"].notna())
plot_meta["max_lr_ratio"] = plot_meta[["lr1_ratio", "lr2_ratio", "lr3_ratio"]].max(axis=1)
plot_meta["max_lr_p2p_uv"] = plot_meta[["lr1_p2p_uv", "lr2_p2p_uv", "lr3_p2p_uv"]].max(axis=1)
plot_meta["max_lr_prominence_uv"] = plot_meta[["lr1_prominence_uv", "lr2_prominence_uv", "lr3_prominence_uv"]].max(axis=1)
plot_meta["max_lr_peak_uv"] = plot_meta[["lr1_peak_uv", "lr2_peak_uv", "lr3_peak_uv"]].max(axis=1)

mask = plot_meta["has_ER"] & plot_meta["has_LR"] & (plot_meta["confidence_score"] >= min_confidence_score_to_plot)
if not show_reject_mr_like:
    mask &= ~plot_meta["reject_mr_like"]

rank_cols = ["max_lr_prominence_uv", "max_lr_peak_uv"]
top_meta = (
    plot_meta.loc[mask].replace([np.inf, -np.inf], np.nan)
    .dropna(subset=rank_cols, how="all")
    .sort_values(rank_cols, ascending=[False, False], na_position="last")
    .head(n_plots).copy()
)
print("Эпох для графиков:", len(top_meta))
print("Ранжирование: LR prominence, then LR amplitude")

display_cols = [
    "epoch_index", "stim_time_s", "er_p2p_uv", "mr_peak_ms", "mr_er_ratio", "reject_mr_like", "n_lr",
    "lr1_peak_ms", "lr1_duration_ms", "lr1_ratio", "lr1_p2p_uv", "lr2_peak_ms", "lr2_duration_ms", "lr2_ratio", "lr2_p2p_uv", "confidence_score",
]
display(top_meta[[col for col in display_cols if col in top_meta.columns]])

if len(top_meta) == 0:
    print("Нет эпох с ER и LR для построения")
else:
    fig, axes = plt.subplots(len(top_meta), 1, figsize=(7.5, max(1.35 * len(top_meta), 4.5)), sharex=True, squeeze=False)
    axes = axes[:, 0]

    for plot_idx, (_, row) in enumerate(top_meta.iterrows()):
        ax = axes[plot_idx]
        epoch_idx = int(row["epoch_index"])
        signal = epochs_data_uv[epoch_idx]
        ax.plot(times_ms, signal, color="black", lw=1.0)
        ax.axvline(0, color="black", linestyle=":", lw=0.8, alpha=0.6)

        spans = [
            (row.get("er_start_ms", np.nan), row.get("er_end_ms", np.nan), "green", 0.18, "ER region"),
            (row.get("mr_start_ms", np.nan), row.get("mr_end_ms", np.nan), "orange", 0.12, "MR region"),
        ]
        for start_ms, end_ms, color, alpha, label in spans:
            if pd.notna(start_ms) and pd.notna(end_ms):
                ax.axvspan(float(start_ms), float(end_ms), color=color, alpha=alpha, label=label if plot_idx == 0 else None)

        lr_peaks = []
        for k in [1, 2, 3]:
            start_ms, end_ms, peak_ms = [row.get(f"lr{k}_{col}", np.nan) for col in ["start_ms", "end_ms", "peak_ms"]]
            if pd.notna(start_ms) and pd.notna(end_ms):
                ax.axvspan(float(start_ms), float(end_ms), color="blue", alpha=0.20, label="LR region" if plot_idx == 0 and k == 1 else None)
            if pd.notna(peak_ms):
                lr_peaks.append(float(peak_ms))
                ax.axvline(float(peak_ms), color="blue", lw=0.8, alpha=0.7)

        if lr_peaks:
            ax.axvline(min(lr_peaks), color="gray", linestyle="--", lw=1.0, alpha=0.8, label="Earliest LR onset" if plot_idx == 0 else None)

        ax.set_title(f"epoch {epoch_idx}", fontsize=8, loc="left", pad=2)
        ax.grid(True, linestyle=":", alpha=0.35)
        ax.tick_params(axis="both", labelsize=7, pad=2)
        ax.margins(y=0.18)

    axes[-1].set_xlabel("Time, ms")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, fontsize=7, loc="upper left", ncol=4, bbox_to_anchor=(0.13, 0.965), frameon=False)
    fig.text(0.015, 0.5, f"Amplitude, {amplitude_unit}", va="center", rotation="vertical", fontsize=9)
    fig.suptitle(f"{globals().get('ch_name', 'EMG')} Channel Epochs", fontsize=12, weight="bold")
    fig.subplots_adjust(left=0.13, right=0.98, bottom=0.07, top=0.91, hspace=0.55)
    plt.show()


Эпох для графиков: 10
Ранжирование: LR prominence, then LR amplitude


,epoch_index,stim_time_s,er_p2p_uv,mr_peak_ms,mr_er_ratio,reject_mr_like,n_lr,lr1_peak_ms,lr1_duration_ms,lr1_ratio,lr1_p2p_uv,lr2_peak_ms,lr2_duration_ms,lr2_ratio,lr2_p2p_uv,confidence_score
188,188,NaN,1.434375,284.50,0.629847,False,1,474.75,13.75,1.815469,2.604063,NaN,NaN,NaN,NaN,1
55,55,NaN,1.426250,236.25,2.114373,False,1,379.00,19.00,0.807844,1.152188,NaN,NaN,NaN,NaN,1
353,353,NaN,1.424062,266.75,0.747641,False,1,381.00,14.25,0.518104,0.737813,NaN,NaN,NaN,NaN,1
508,508,NaN,3.219687,231.75,0.231874,False,1,451.75,12.50,0.186353,0.600000,NaN,NaN,NaN,NaN,1
343,343,NaN,1.794062,284.50,0.635778,False,1,344.75,13.75,0.592580,1.063125,NaN,NaN,NaN,NaN,1
384,384,NaN,2.075313,233.75,0.458365,False,1,426.25,12.75,0.310345,0.644062,NaN,NaN,NaN,NaN,1
390,390,NaN,2.441875,274.25,0.141797,False,1,311.25,12.75,0.314820,0.768750,NaN,NaN,NaN,NaN,1
357,357,NaN,1.515000,252.25,0.442450,False,1,346.00,10.00,0.228548,0.346250,NaN,NaN,NaN,NaN,1
231,231,NaN,0.873125,248.75,0.823908,False,1,362.25,14.75,0.496063,0.433125,NaN,NaN,NaN,NaN,1
522,522,NaN,3.224375,252.00,0.462202,False,1,370.75,26.75,0.193545,0.624062,NaN,NaN,NaN,NaN,1
